In [46]:
import numpy as np
import pandas as pd
import mne
import math

In [49]:
offline_full = pd.read_csv("thetaEEG_full_2back_-1580742246_20260617_154844.csv")
offline_full

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,Ch07,Ch08
0,0.000,199999.968750,199999.96875,199999.96875,199999.968750,321.745941,199999.96875,119156.062500,264.954590
1,0.004,199999.968750,199999.96875,199999.96875,199999.968750,320.887604,199999.96875,119781.359375,263.977051
2,0.008,199999.968750,199999.96875,199999.96875,199999.968750,320.720703,199999.96875,119223.046875,267.791748
3,0.012,199999.968750,199999.96875,199999.96875,199999.968750,323.677094,199999.96875,118403.984375,266.289734
4,0.016,199999.968750,199999.96875,199999.96875,199999.968750,320.529968,199999.96875,118835.437500,266.218201
...,...,...,...,...,...,...,...,...,...
25913,103.652,-10796.262695,199999.96875,199999.96875,175374.593750,71049.375000,199999.96875,199999.968750,817.585144
25914,103.656,-10788.942383,199999.96875,199999.96875,175235.718750,71229.796875,199999.96875,199999.968750,409.770020
25915,103.660,-10791.349609,199999.96875,199999.96875,175259.093750,71825.046875,199999.96875,199999.968750,4545.975098
25916,103.664,-10478.307617,199999.96875,199999.96875,175936.546875,72038.171875,199999.96875,199999.968750,467.276611


In [70]:
import numpy as np
import pandas as pd
from scipy.signal import iirnotch, butter, filtfilt

# -----------------------------
# 60 Hz Notch Filter
# -----------------------------
def notch_filter(data, fs=250, freq=60, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, data)

# -----------------------------
# 1–30 Hz Bandpass Filter
# -----------------------------
def bandpass_filter(data, fs=250, low=1, high=30, order=4):
    nyq = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

# -----------------------------
# Combined EEG Preprocessing
# -----------------------------
def preprocess_eeg(df, col=" Ch08", fs=250):
    raw = df[col].values

    # Apply filters
    notched = notch_filter(raw, fs=fs)
    bandpassed = bandpass_filter(notched, fs=fs)

    # Store results
    df[col + "_notched"] = notched
    df[col + "_bp"] = bandpassed
    return df
df = preprocess_eeg(offline_full, col=" Ch08", fs=250)

In [71]:
df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,Ch07,Ch08,Ch08_notched,Ch08_bp
0,0.000,199999.968750,199999.96875,199999.96875,199999.968750,321.745941,199999.96875,119156.062500,264.954590,265.347874,-0.256586
1,0.004,199999.968750,199999.96875,199999.96875,199999.968750,320.887604,199999.96875,119781.359375,263.977051,264.567648,0.053671
2,0.008,199999.968750,199999.96875,199999.96875,199999.968750,320.720703,199999.96875,119223.046875,267.791748,267.454490,0.149223
3,0.012,199999.968750,199999.96875,199999.96875,199999.968750,323.677094,199999.96875,118403.984375,266.289734,265.642409,-0.065661
4,0.016,199999.968750,199999.96875,199999.96875,199999.968750,320.529968,199999.96875,118835.437500,266.218201,266.488563,-0.515981
...,...,...,...,...,...,...,...,...,...,...,...
25913,103.652,-10796.262695,199999.96875,199999.96875,175374.593750,71049.375000,199999.96875,199999.968750,817.585144,824.999460,1944.037064
25914,103.656,-10788.942383,199999.96875,199999.96875,175235.718750,71229.796875,199999.96875,199999.968750,409.770020,478.517688,1806.071654
25915,103.660,-10791.349609,199999.96875,199999.96875,175259.093750,71825.046875,199999.96875,199999.968750,4545.975098,4549.864724,1401.751436
25916,103.664,-10478.307617,199999.96875,199999.96875,175936.546875,72038.171875,199999.96875,199999.968750,467.276611,396.202354,735.204756


In [67]:
# Step 1: threshold crossings
trigger_on = trigger_channel[trigger_channel[" Ch08"] > 4500].copy()

# Step 2: enforce 3‑second spacing
MIN_SPACING = 7.0  # seconds
kept = []
last_time = 0   # ensures first event is always kept

for t in trigger_on["Time"]:
    if t - last_time >= MIN_SPACING:
        kept.append(t)
        last_time = t

# Step 3: build filtered DataFrame
trigger_on_spaced = trigger_on[trigger_on["Time"].isin(kept)]
trigger_on_spaced


,Time,Ch08
20196,80.784,4507.518066
22048,88.192,4540.349121
24060,96.240,4540.610840
25815,103.260,4548.073242


In [13]:
lifu_on_ts = np.array([313249.3567498, 313256.5596819, 313263.7622901, 313271.1735891,313277.7814648, 313284.8678493])
lifu_on_ts

array([313249.3567498, 313256.5596819, 313263.7622901, 313271.1735891,
       313277.7814648, 313284.8678493])

In [45]:
pre = 1
post = 0
windows = []
event = np.array([66.1, 73.3, 80.5, 87.9, 94.5])

for idx, ev in enumerate(event):
    mask = (
        (trigger_channel['Time'] >= ev - pre) &
        (trigger_channel['Time'] <= ev + post)
    )
    
    window_df = trigger_channel.loc[mask].copy()
    window_df["t_rel"] = window_df["Time"] - ev
    window_df["idx"] = idx
    windows.append(window_df)


# for idx, event in enumerate(lifu_on_ts):
#     mask = (
#         (trigger_channel['Time'] >= (event - pre)) &
#         (trigger_channel['Time'] <= (event + post))
#     )
    
#     window_df = trigger_channel.loc[mask].copy()
#     window_df["t_rel"] = window_df["Time"] - event
#     window_df["idx"] = idx
#     windows.append(window_df)


In [44]:
windows[0]

,Ch08,Time,t_rel,idx
16025,350.379974,64.100,-2.000,0
16026,351.977386,64.104,-1.996,0
16027,350.666107,64.108,-1.992,0
16028,350.284607,64.112,-1.988,0
16029,350.856842,64.116,-1.984,0
...,...,...,...,...
17771,376.606049,71.084,4.984,0
17772,379.586273,71.088,4.988,0
17773,380.039246,71.092,4.992,0
17774,377.392761,71.096,4.996,0


In [187]:
pre = 2
post = 5
windows = []
for idx, event in enumerate(event_times): 
    mask = (raw['Time'] >= (event - pre)) & (raw['Time'] <= (event + post))
    window_df = raw.loc[mask].copy()
    window_df["t_rel"] = window_df["Time"] - event 
    window_df["idx"] = idx
    windows.append(window_df)
    

ValueError: picks ('Time') could not be interpreted as channel names (no channel "['Time']"), channel types (no type "Time" present), or a generic type (just "all" or "data")

In [ ]:
windows_df = pd.concat(windows, ignore_index=True)
windows_df_0 = windows_df[windows_df['idx']==0]
windows_df_0

In [ ]:
mask = merged_df[